# Análise de variância (ANOVA) com um fator

**Disciplina:** Análise de Dados para Decisões em Negócios (graduação em Administração)

**Objetivos de aprendizagem.** Ao final desta aula, você será capaz de:

1. explicar por que a comparação de três ou mais médias pede a ANOVA, e não uma sequência de testes t;
2. formular as hipóteses nula e alternativa da ANOVA com um fator;
3. calcular, passo a passo, as somas de quadrados, os quadrados médios e a estatística F;
4. aplicar o teste com a função `f_oneway` da biblioteca SciPy e verificar os pressupostos com `levene` e `shapiro`;
5. identificar quais grupos diferem entre si com o teste de Tukey (`tukey_hsd`);
6. interpretar o valor-p e traduzir o resultado em uma recomendação gerencial.

**Como usar este notebook.** Digite o código linha a linha, junto com o professor, e execute cada célula com `Shift + Enter`. Todos os dados são fictícios e foram criados apenas para fins didáticos.

## Revisão rápida dos conceitos

Nas duas últimas aulas, comparamos **duas** médias com o teste t. Muitas decisões de negócios, porém, envolvem três ou mais alternativas: três campanhas promocionais, quatro fornecedores, três turnos de produção. A **análise de variância (ANOVA) com um fator** compara, em um único teste, as médias de $k$ grupos independentes ($k \ge 3$). O "fator" é a variável que define os grupos (tipo de promoção, fornecedor, turno), e cada grupo é um **nível** desse fator.

**Por que não fazer vários testes t?** Com três grupos, seriam três comparações (1 com 2, 1 com 3 e 2 com 3); com quatro grupos, seis. Cada teste feito com $\alpha = 5\%$ tem 5% de chance de apontar uma diferença que não existe. Se as comparações fossem independentes, a chance de cometer pelo menos um desses erros em três testes seria $1 - 0{,}95^3 \approx 14{,}3\%$, quase o triplo do nível planejado. A ANOVA mantém o risco total em $\alpha$ porque faz uma única pergunta: as médias são todas iguais?

**Hipóteses.** Com $\mu_1, \mu_2, \ldots, \mu_k$ representando as médias populacionais dos grupos:

* $H_0$: $\mu_1 = \mu_2 = \cdots = \mu_k$ (todas as médias são iguais);
* $H_1$: pelo menos uma média é diferente das demais.

Note que $H_1$ **não** afirma que todas as médias são diferentes. Por isso, quando rejeitamos $H_0$, precisamos de uma etapa adicional, as **comparações múltiplas**, para descobrir quais grupos diferem.

**A ideia central.** A ANOVA separa a variação total dos dados em duas partes:

* a variação **entre os grupos**, que mede o quanto as médias dos grupos se afastam da média geral;
* a variação **dentro dos grupos**, que mede o quanto os valores se afastam da média do próprio grupo (a variação natural, que existiria mesmo sem diferença entre os grupos).

Se $H_0$ for verdadeira, as duas variações, depois de divididas pelos respectivos graus de liberdade, devem ter tamanho parecido. Se a variação entre os grupos for muito maior que a variação dentro deles, temos evidência de que as médias não são todas iguais.

**As contas.** Sejam $n_i$, $\bar{x}_i$ e $s_i$ o tamanho, a média e o desvio-padrão do grupo $i$, $\bar{\bar{x}}$ a média geral (de todos os valores juntos) e $N$ o total de observações.

| Componente | Soma de quadrados | Graus de liberdade | Quadrado médio |
|---|---|---|---|
| Entre os grupos | $SQ_{entre} = \sum n_i (\bar{x}_i - \bar{\bar{x}})^2$ | $k - 1$ | $QM_{entre} = SQ_{entre} / (k-1)$ |
| Dentro dos grupos | $SQ_{dentro} = \sum (n_i - 1)\, s_i^2$ | $N - k$ | $QM_{dentro} = SQ_{dentro} / (N-k)$ |
| Total | $SQ_{total} = SQ_{entre} + SQ_{dentro}$ | $N - 1$ | |

A estatística de teste é

$$F = \frac{QM_{entre}}{QM_{dentro}}.$$

Sob $H_0$, ela segue a distribuição F com $k-1$ graus de liberdade no numerador e $N-k$ no denominador. O teste é sempre **unilateral à direita**: só valores grandes de F contrariam $H_0$. No livro de Anderson et al., as mesmas quantidades aparecem com os nomes SQTR e QMTR (tratamentos) e SQE e QME (erro).

**Tamanho do efeito.** A razão $\eta^2 = SQ_{entre} / SQ_{total}$ (lê-se "eta ao quadrado") indica a proporção da variação total explicada pela diferença entre os grupos. Ela ajuda a avaliar se uma diferença estatisticamente significativa também é relevante na prática.

**Pressupostos.**

1. As amostras são aleatórias e independentes entre si (cada observação pertence a um único grupo).
2. Em cada população, a variável tem distribuição aproximadamente normal. Verificaremos com o teste de Shapiro-Wilk (`shapiro`).
3. As populações têm a mesma variância (homocedasticidade). Verificaremos com o teste de Levene (`levene`).

Nos dois testes de pressupostos, a hipótese nula é a de que o pressuposto vale. Assim, um valor-p **maior** que 5% é o resultado desejado. Com amostras pequenas, como as desta aula, esses testes têm pouca capacidade de detectar violações, e por isso convém olhar também os desvios-padrão e o desenho da coleta.

**Regra de decisão.** Fixado $\alpha$ (em geral, 5%), rejeitamos $H_0$ quando o valor-p for menor que $\alpha$ ou, de forma equivalente, quando o F calculado for maior que o F crítico.

**Comparações múltiplas: o teste de Tukey.** Se rejeitarmos $H_0$, o teste HSD (*honestly significant difference*) de Tukey compara todos os pares de grupos e ajusta os valores-p para que o risco total de erro continue em $\alpha$.

## Passo zero: um pouco de Python

Nesta aula, vamos usar os mesmos recursos das aulas anteriores (listas, `len()`, `print()`, `**` e `if`/`else`) e mais um: o operador `+` aplicado a listas, que **junta** duas ou mais listas em uma só. Ele será útil para calcular a média geral. Execute a célula abaixo para testar.

In [1]:
loja_a = [10, 20] # Uma lista com dois valores.
loja_b = [30, 40, 50] # Outra lista com três valores.
todas = loja_a + loja_b # Junta as duas listas.
print(todas) # Mostra a lista combinada.
print(len(todas)) # Mostra quantos itens há na lista combinada.

[10, 20, 30, 40, 50]
5


Agora vamos carregar as ferramentas que usaremos na aula:

* `statistics`, para a média e o desvio-padrão;
* `stats`, da biblioteca SciPy, para a ANOVA (`f_oneway`), os testes de pressupostos (`levene` e `shapiro`), o teste de Tukey (`tukey_hsd`) e a distribuição F.

In [2]:
import statistics # Média e desvio-padrão.
from scipy import stats # ANOVA, pressupostos, Tukey e distribuição F.

---
## Exemplo 1. Três tipos de promoção em supermercados

Uma rede de supermercados quer escolher o formato de promoção para o lançamento de um sabão líquido. Durante duas semanas, 19 lojas de porte semelhante foram sorteadas para três formatos: **desconto** de 15% no preço (6 lojas), **leve 3, pague 2** (7 lojas) e **brinde** de um amaciante (6 lojas). Foi registrada a venda média diária do produto, em unidades, em cada loja.

| Grupo | n | Valores |
|---|---|---|
| Desconto de 15% | 6 | 42; 45; 39; 47; 44; 41 |
| Leve 3, pague 2 | 7 | 48; 52; 50; 46; 53; 49; 51 |
| Brinde | 6 | 40; 43; 38; 42; 44; 39 |

**Pergunta gerencial.** As vendas médias diárias são **diferentes** entre os três formatos de promoção?

**Hipóteses** ($k = 3$ grupos):

* $H_0$: $\mu_{\text{desconto}} = \mu_{\text{leve3}} = \mu_{\text{brinde}}$ (os três formatos vendem, em média, o mesmo)
* $H_1$: pelo menos uma média é diferente (pelo menos um formato tem venda média diferente)

Teste F (unilateral à direita), com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em listas, uma para cada grupo, e mostrar a média de cada uma.

In [3]:
desconto = [42, 45, 39, 47, 44, 41]
leve3 = [48, 52, 50, 46, 53, 49, 51]
brinde = [40, 43, 38, 42, 44, 39]

print("Média de desconto:", round(statistics.mean(desconto), 4))
print("Média de leve3:", round(statistics.mean(leve3), 4))
print("Média de brinde:", round(statistics.mean(brinde), 4))

Média de desconto: 43
Média de leve3: 49.8571
Média de brinde: 41


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo e a média geral. Para a média geral, juntamos todas as listas com o operador `+`.

In [4]:
n1 = len(desconto) # Tamanho do grupo 1.
n2 = len(leve3) # Tamanho do grupo 2.
n3 = len(brinde) # Tamanho do grupo 3.
media1 = statistics.mean(desconto) # Média do grupo 1.
media2 = statistics.mean(leve3) # Média do grupo 2.
media3 = statistics.mean(brinde) # Média do grupo 3.
dp1 = statistics.stdev(desconto) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(leve3) # Desvio-padrão amostral do grupo 2.
dp3 = statistics.stdev(brinde) # Desvio-padrão amostral do grupo 3.

todos = desconto + leve3 + brinde # Junta todas as observações.
N = len(todos) # Total de observações.
k = 3 # Número de grupos.
media_geral = statistics.mean(todos) # Média geral.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))
print("Grupo 3: n =", n3, "| média =", round(media3, 4), "| desvio-padrão =", round(dp3, 4))
print("Média geral:", round(media_geral, 4), "| N =", N)

Grupo 1: n = 6 | média = 43 | desvio-padrão = 2.8983
Grupo 2: n = 7 | média = 49.8571 | desvio-padrão = 2.4103
Grupo 3: n = 6 | média = 41 | desvio-padrão = 2.3664
Média geral: 44.8947 | N = 19


**Passo 3.** Calcular a estatística F à mão, usando as fórmulas da tabela da ANOVA.

In [5]:
sq_entre = n1 * (media1 - media_geral) ** 2 + n2 * (media2 - media_geral) ** 2 + n3 * (media3 - media_geral) ** 2 # Variação entre os grupos.
sq_dentro = (n1 - 1) * dp1 ** 2 + (n2 - 1) * dp2 ** 2 + (n3 - 1) * dp3 ** 2 # Variação dentro dos grupos.
gl_entre = k - 1 # Graus de liberdade do numerador.
gl_dentro = N - k # Graus de liberdade do denominador.
qm_entre = sq_entre / gl_entre # Quadrado médio entre os grupos.
qm_dentro = sq_dentro / gl_dentro # Quadrado médio dentro dos grupos.
f_calculado = qm_entre / qm_dentro # Estatística F.
eta2 = sq_entre / (sq_entre + sq_dentro) # Proporção da variação explicada pelos grupos.

print("SQ entre:", round(sq_entre, 4), "| gl =", gl_entre, "| QM entre:", round(qm_entre, 4))
print("SQ dentro:", round(sq_dentro, 4), "| gl =", gl_dentro, "| QM dentro:", round(qm_dentro, 4))
print("F calculado:", round(f_calculado, 4))
print("Eta ao quadrado:", round(eta2, 4))

SQ entre: 284.9323 | gl = 2 | QM entre: 142.4662
SQ dentro: 104.8571 | gl = 16 | QM dentro: 6.5536
F calculado: 21.7387
Eta ao quadrado: 0.731


**Passo 4.** Conferir com a função `f_oneway`, que faz toda a conta de uma vez. Basta informar as listas, separadas por vírgulas. Valores-p muito pequenos aparecem em notação científica: `2.742e-05`, por exemplo, significa $2{,}742 \times 10^{-5}$, ou seja, 0,00002742.

In [6]:
resultado = stats.f_oneway(desconto, leve3, brinde)

print("F (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 8))

F (SciPy): 21.7387
valor-p: 2.742e-05


**Passo 5.** Verificar os pressupostos. No teste de Levene, $H_0$ é a igualdade das variâncias; no de Shapiro-Wilk, $H_0$ é a normalidade dos dados do grupo. Valores-p maiores que 0,05 indicam que não há evidência contra o pressuposto.

In [7]:
levene = stats.levene(desconto, leve3, brinde) # Homogeneidade das variâncias.
print("Levene: valor-p =", round(levene.pvalue, 4))

print("Shapiro-Wilk (desconto): valor-p =", round(stats.shapiro(desconto).pvalue, 4))
print("Shapiro-Wilk (leve3): valor-p =", round(stats.shapiro(leve3).pvalue, 4))
print("Shapiro-Wilk (brinde): valor-p =", round(stats.shapiro(brinde).pvalue, 4))

Levene: valor-p = 0.7816
Shapiro-Wilk (desconto): valor-p = 0.985
Shapiro-Wilk (leve3): valor-p = 0.976
Shapiro-Wilk (brinde): valor-p = 0.7393


**Passo 6.** Comparar com o valor crítico e tomar a decisão.

In [8]:
alfa = 0.05
f_critico = stats.f.ppf(1 - alfa, gl_entre, gl_dentro) # O teste F é sempre unilateral à direita.
print("F crítico:", round(f_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0. Pelo menos uma média é diferente.")
else:
    print("Decisão: não rejeitamos H0.")

F crítico: 3.6337
Decisão: rejeitamos H0. Pelo menos uma média é diferente.


**Passo 7.** Como rejeitamos $H_0$, aplicamos o teste de Tukey para descobrir quais pares de grupos diferem. Na saída, os grupos são numerados a partir de zero, na ordem em que as listas foram informadas: 0 = `desconto`, 1 = `leve3`, 2 = `brinde`. Cada par aparece duas vezes, com sinais trocados; basta ler uma das linhas. A coluna `p-value` mostra três casas decimais, e `0.000` indica valor-p menor que 0,0005.

In [9]:
tukey = stats.tukey_hsd(desconto, leve3, brinde)
print(tukey)

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -6.857     0.001   -10.532    -3.182
 (0 - 2)      2.000     0.388    -1.814     5.814
 (1 - 0)      6.857     0.001     3.182    10.532
 (1 - 2)      8.857     0.000     5.182    12.532
 (2 - 0)     -2.000     0.388    -5.814     1.814
 (2 - 1)     -8.857     0.000   -12.532    -5.182



**Leitura do Tukey.** Leve 3, pague 2 vende cerca de 6,86 unidades a mais que o desconto (valor-p ≈ 0,0005) e 8,86 a mais que o brinde (valor-p < 0,0001). A diferença de 2 unidades entre desconto e brinde não é significativa (valor-p ≈ 0,39).

**Interpretação gerencial.** As lojas com a promoção leve 3, pague 2 venderam, em média, 49,86 unidades por dia, contra 43,00 nas lojas com desconto e 41,00 nas lojas com brinde ($F = 21{,}74$ com 2 e 16 graus de liberdade; valor-p ≈ 0,00003). Rejeitamos $H_0$: os formatos não têm o mesmo desempenho. O $\eta^2$ de 0,73 indica que 73% da variação das vendas entre as lojas está associada ao formato da promoção. Os pressupostos foram atendidos (Levene: valor-p ≈ 0,78; Shapiro-Wilk com valores-p acima de 0,70). O teste de Tukey mostra que o leve 3, pague 2 supera os outros dois formatos, enquanto desconto e brinde não diferem entre si. Antes de adotar o leve 3, pague 2 em toda a rede, o gerente deve comparar a **margem** de cada formato, porque vender mais unidades com um desconto efetivo de 33% pode render menos lucro do que vender um pouco menos com desconto de 15%.

---
## Exemplo 2. Prazo de entrega de três transportadoras

Uma loja virtual de cosméticos trabalha com três transportadoras para entregas na Região Metropolitana de Belo Horizonte. O coordenador de logística sorteou pedidos de cada transportadora e registrou o prazo de entrega, em dias.

| Grupo | n | Valores |
|---|---|---|
| Transportadora A | 7 | 3,2; 2,8; 3,5; 3,0; 2,9; 3,4; 3,1 |
| Transportadora B | 6 | 4,1; 3,8; 4,4; 3,9; 4,2; 4,0 |
| Transportadora C | 7 | 3,6; 3,3; 3,9; 3,4; 3,7; 3,5; 3,8 |

**Pergunta gerencial.** Os prazos médios de entrega das três transportadoras são **diferentes**?

**Hipóteses** ($k = 3$ grupos):

* $H_0$: $\mu_{\text{A}} = \mu_{\text{B}} = \mu_{\text{C}}$ (as três transportadoras têm o mesmo prazo médio)
* $H_1$: pelo menos uma média é diferente (pelo menos uma transportadora tem prazo médio diferente)

Teste F (unilateral à direita), com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em listas, uma para cada grupo, e mostrar a média de cada uma. Lembre-se de que, em Python, o separador decimal é o ponto.

In [10]:
transp_a = [3.2, 2.8, 3.5, 3.0, 2.9, 3.4, 3.1]
transp_b = [4.1, 3.8, 4.4, 3.9, 4.2, 4.0]
transp_c = [3.6, 3.3, 3.9, 3.4, 3.7, 3.5, 3.8]

print("Média de transp_a:", round(statistics.mean(transp_a), 4))
print("Média de transp_b:", round(statistics.mean(transp_b), 4))
print("Média de transp_c:", round(statistics.mean(transp_c), 4))

Média de transp_a: 3.1286
Média de transp_b: 4.0667
Média de transp_c: 3.6


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo e a média geral. Para a média geral, juntamos todas as listas com o operador `+`.

In [11]:
n1 = len(transp_a) # Tamanho do grupo 1.
n2 = len(transp_b) # Tamanho do grupo 2.
n3 = len(transp_c) # Tamanho do grupo 3.
media1 = statistics.mean(transp_a) # Média do grupo 1.
media2 = statistics.mean(transp_b) # Média do grupo 2.
media3 = statistics.mean(transp_c) # Média do grupo 3.
dp1 = statistics.stdev(transp_a) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(transp_b) # Desvio-padrão amostral do grupo 2.
dp3 = statistics.stdev(transp_c) # Desvio-padrão amostral do grupo 3.

todos = transp_a + transp_b + transp_c # Junta todas as observações.
N = len(todos) # Total de observações.
k = 3 # Número de grupos.
media_geral = statistics.mean(todos) # Média geral.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))
print("Grupo 3: n =", n3, "| média =", round(media3, 4), "| desvio-padrão =", round(dp3, 4))
print("Média geral:", round(media_geral, 4), "| N =", N)

Grupo 1: n = 7 | média = 3.1286 | desvio-padrão = 0.2563
Grupo 2: n = 6 | média = 4.0667 | desvio-padrão = 0.216
Grupo 3: n = 7 | média = 3.6 | desvio-padrão = 0.216
Média geral: 3.575 | N = 20


**Passo 3.** Calcular a estatística F à mão, usando as fórmulas da tabela da ANOVA.

In [12]:
sq_entre = n1 * (media1 - media_geral) ** 2 + n2 * (media2 - media_geral) ** 2 + n3 * (media3 - media_geral) ** 2 # Variação entre os grupos.
sq_dentro = (n1 - 1) * dp1 ** 2 + (n2 - 1) * dp2 ** 2 + (n3 - 1) * dp3 ** 2 # Variação dentro dos grupos.
gl_entre = k - 1 # Graus de liberdade do numerador.
gl_dentro = N - k # Graus de liberdade do denominador.
qm_entre = sq_entre / gl_entre # Quadrado médio entre os grupos.
qm_dentro = sq_dentro / gl_dentro # Quadrado médio dentro dos grupos.
f_calculado = qm_entre / qm_dentro # Estatística F.
eta2 = sq_entre / (sq_entre + sq_dentro) # Proporção da variação explicada pelos grupos.

print("SQ entre:", round(sq_entre, 4), "| gl =", gl_entre, "| QM entre:", round(qm_entre, 4))
print("SQ dentro:", round(sq_dentro, 4), "| gl =", gl_dentro, "| QM dentro:", round(qm_dentro, 4))
print("F calculado:", round(f_calculado, 4))
print("Eta ao quadrado:", round(eta2, 4))

SQ entre: 2.8499 | gl = 2 | QM entre: 1.4249
SQ dentro: 0.9076 | gl = 17 | QM dentro: 0.0534
F calculado: 26.6896
Eta ao quadrado: 0.7585


**Passo 4.** Conferir com a função `f_oneway`, que faz toda a conta de uma vez. Basta informar as listas, separadas por vírgulas. Valores-p muito pequenos aparecem em notação científica: `2.742e-05`, por exemplo, significa $2{,}742 \times 10^{-5}$, ou seja, 0,00002742.

In [13]:
resultado = stats.f_oneway(transp_a, transp_b, transp_c)

print("F (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 8))

F (SciPy): 26.6896
valor-p: 5.7e-06


**Passo 5.** Verificar os pressupostos. No teste de Levene, $H_0$ é a igualdade das variâncias; no de Shapiro-Wilk, $H_0$ é a normalidade dos dados do grupo. Valores-p maiores que 0,05 indicam que não há evidência contra o pressuposto.

In [14]:
levene = stats.levene(transp_a, transp_b, transp_c) # Homogeneidade das variâncias.
print("Levene: valor-p =", round(levene.pvalue, 4))

print("Shapiro-Wilk (transp_a): valor-p =", round(stats.shapiro(transp_a).pvalue, 4))
print("Shapiro-Wilk (transp_b): valor-p =", round(stats.shapiro(transp_b).pvalue, 4))
print("Shapiro-Wilk (transp_c): valor-p =", round(stats.shapiro(transp_c).pvalue, 4))

Levene: valor-p = 0.8688
Shapiro-Wilk (transp_a): valor-p = 0.8366
Shapiro-Wilk (transp_b): valor-p = 0.9637
Shapiro-Wilk (transp_c): valor-p = 0.9493


**Passo 6.** Comparar com o valor crítico e tomar a decisão.

In [15]:
alfa = 0.05
f_critico = stats.f.ppf(1 - alfa, gl_entre, gl_dentro) # O teste F é sempre unilateral à direita.
print("F crítico:", round(f_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0. Pelo menos uma média é diferente.")
else:
    print("Decisão: não rejeitamos H0.")

F crítico: 3.5915
Decisão: rejeitamos H0. Pelo menos uma média é diferente.


**Passo 7.** Como rejeitamos $H_0$, aplicamos o teste de Tukey para descobrir quais pares de grupos diferem. Na saída, os grupos são numerados a partir de zero, na ordem em que as listas foram informadas: 0 = `transp_a`, 1 = `transp_b`, 2 = `transp_c`. Cada par aparece duas vezes, com sinais trocados; basta ler uma das linhas. A coluna `p-value` mostra três casas decimais, e `0.000` indica valor-p menor que 0,0005.

In [16]:
tukey = stats.tukey_hsd(transp_a, transp_b, transp_c)
print(tukey)

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.938     0.000    -1.268    -0.608
 (0 - 2)     -0.471     0.004    -0.788    -0.155
 (1 - 0)      0.938     0.000     0.608     1.268
 (1 - 2)      0.467     0.006     0.137     0.796
 (2 - 0)      0.471     0.004     0.155     0.788
 (2 - 1)     -0.467     0.006    -0.796    -0.137



**Leitura do Tukey.** Todos os pares diferem ao nível de 5%: B é mais lenta que A em cerca de 0,94 dia (valor-p < 0,0001) e que C em cerca de 0,47 dia (valor-p ≈ 0,006); C é mais lenta que A em cerca de 0,47 dia (valor-p ≈ 0,004).

**Interpretação gerencial.** Os prazos médios foram de 3,13 dias para a transportadora A, 4,07 dias para a B e 3,60 dias para a C ($F = 26{,}69$ com 2 e 17 graus de liberdade; valor-p ≈ 0,000006). Rejeitamos $H_0$ e, pelo teste de Tukey, as três transportadoras diferem entre si, com A como a mais rápida. A diferença entre A e B é de quase um dia por pedido, o que pode afetar a satisfação dos clientes e o volume de reclamações. A recomendação é concentrar os pedidos urgentes na transportadora A, desde que o custo do frete e a taxa de avarias também sejam favoráveis, e renegociar o nível de serviço com a transportadora B.

---
## Exemplo 3. Satisfação de clientes em três unidades de um restaurante

Uma rede de restaurantes de Belo Horizonte aplicou uma pesquisa de satisfação, com notas de 0 a 10, a clientes sorteados em três unidades: Savassi, Pampulha e Buritis.

| Grupo | n | Valores |
|---|---|---|
| Savassi | 8 | 8,1; 7,6; 8,4; 7,9; 8,2; 7,7; 8,0; 7,8 |
| Pampulha | 7 | 7,9; 8,3; 7,5; 8,0; 7,7; 8,2; 7,8 |
| Buritis | 8 | 8,2; 7,8; 7,6; 8,3; 7,9; 8,1; 7,7; 8,0 |

**Pergunta gerencial.** A satisfação média dos clientes é **diferente** entre as três unidades?

**Hipóteses** ($k = 3$ grupos):

* $H_0$: $\mu_{\text{savassi}} = \mu_{\text{pampulha}} = \mu_{\text{buritis}}$ (a satisfação média é igual nas três unidades)
* $H_1$: pelo menos uma média é diferente (pelo menos uma unidade tem satisfação média diferente)

Teste F (unilateral à direita), com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em listas, uma para cada grupo, e mostrar a média de cada uma.

In [17]:
savassi = [8.1, 7.6, 8.4, 7.9, 8.2, 7.7, 8.0, 7.8]
pampulha = [7.9, 8.3, 7.5, 8.0, 7.7, 8.2, 7.8]
buritis = [8.2, 7.8, 7.6, 8.3, 7.9, 8.1, 7.7, 8.0]

print("Média de savassi:", round(statistics.mean(savassi), 4))
print("Média de pampulha:", round(statistics.mean(pampulha), 4))
print("Média de buritis:", round(statistics.mean(buritis), 4))

Média de savassi: 7.9625
Média de pampulha: 7.9143
Média de buritis: 7.95


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo e a média geral. Para a média geral, juntamos todas as listas com o operador `+`.

In [18]:
n1 = len(savassi) # Tamanho do grupo 1.
n2 = len(pampulha) # Tamanho do grupo 2.
n3 = len(buritis) # Tamanho do grupo 3.
media1 = statistics.mean(savassi) # Média do grupo 1.
media2 = statistics.mean(pampulha) # Média do grupo 2.
media3 = statistics.mean(buritis) # Média do grupo 3.
dp1 = statistics.stdev(savassi) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(pampulha) # Desvio-padrão amostral do grupo 2.
dp3 = statistics.stdev(buritis) # Desvio-padrão amostral do grupo 3.

todos = savassi + pampulha + buritis # Junta todas as observações.
N = len(todos) # Total de observações.
k = 3 # Número de grupos.
media_geral = statistics.mean(todos) # Média geral.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))
print("Grupo 3: n =", n3, "| média =", round(media3, 4), "| desvio-padrão =", round(dp3, 4))
print("Média geral:", round(media_geral, 4), "| N =", N)

Grupo 1: n = 8 | média = 7.9625 | desvio-padrão = 0.2669
Grupo 2: n = 7 | média = 7.9143 | desvio-padrão = 0.2795
Grupo 3: n = 8 | média = 7.95 | desvio-padrão = 0.2449
Média geral: 7.9435 | N = 23


**Passo 3.** Calcular a estatística F à mão, usando as fórmulas da tabela da ANOVA.

In [19]:
sq_entre = n1 * (media1 - media_geral) ** 2 + n2 * (media2 - media_geral) ** 2 + n3 * (media3 - media_geral) ** 2 # Variação entre os grupos.
sq_dentro = (n1 - 1) * dp1 ** 2 + (n2 - 1) * dp2 ** 2 + (n3 - 1) * dp3 ** 2 # Variação dentro dos grupos.
gl_entre = k - 1 # Graus de liberdade do numerador.
gl_dentro = N - k # Graus de liberdade do denominador.
qm_entre = sq_entre / gl_entre # Quadrado médio entre os grupos.
qm_dentro = sq_dentro / gl_dentro # Quadrado médio dentro dos grupos.
f_calculado = qm_entre / qm_dentro # Estatística F.
eta2 = sq_entre / (sq_entre + sq_dentro) # Proporção da variação explicada pelos grupos.

print("SQ entre:", round(sq_entre, 4), "| gl =", gl_entre, "| QM entre:", round(qm_entre, 4))
print("SQ dentro:", round(sq_dentro, 4), "| gl =", gl_dentro, "| QM dentro:", round(qm_dentro, 4))
print("F calculado:", round(f_calculado, 4))
print("Eta ao quadrado:", round(eta2, 4))

SQ entre: 0.0092 | gl = 2 | QM entre: 0.0046
SQ dentro: 1.3873 | gl = 20 | QM dentro: 0.0694
F calculado: 0.0663
Eta ao quadrado: 0.0066


**Passo 4.** Conferir com a função `f_oneway`, que faz toda a conta de uma vez. Basta informar as listas, separadas por vírgulas. Valores-p muito pequenos aparecem em notação científica: `2.742e-05`, por exemplo, significa $2{,}742 \times 10^{-5}$, ou seja, 0,00002742.

In [20]:
resultado = stats.f_oneway(savassi, pampulha, buritis)

print("F (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 8))

F (SciPy): 0.0663
valor-p: 0.93603899


**Passo 5.** Verificar os pressupostos. No teste de Levene, $H_0$ é a igualdade das variâncias; no de Shapiro-Wilk, $H_0$ é a normalidade dos dados do grupo. Valores-p maiores que 0,05 indicam que não há evidência contra o pressuposto.

In [21]:
levene = stats.levene(savassi, pampulha, buritis) # Homogeneidade das variâncias.
print("Levene: valor-p =", round(levene.pvalue, 4))

print("Shapiro-Wilk (savassi): valor-p =", round(stats.shapiro(savassi).pvalue, 4))
print("Shapiro-Wilk (pampulha): valor-p =", round(stats.shapiro(pampulha).pvalue, 4))
print("Shapiro-Wilk (buritis): valor-p =", round(stats.shapiro(buritis).pvalue, 4))

Levene: valor-p = 0.9761
Shapiro-Wilk (savassi): valor-p = 0.9782
Shapiro-Wilk (pampulha): valor-p = 0.9673
Shapiro-Wilk (buritis): valor-p = 0.9332


**Passo 6.** Comparar com o valor crítico e tomar a decisão.

In [22]:
alfa = 0.05
f_critico = stats.f.ppf(1 - alfa, gl_entre, gl_dentro) # O teste F é sempre unilateral à direita.
print("F crítico:", round(f_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0. Pelo menos uma média é diferente.")
else:
    print("Decisão: não rejeitamos H0.")

F crítico: 3.4928
Decisão: não rejeitamos H0.


**Interpretação gerencial.** As notas médias foram de 7,96 na Savassi, 7,91 na Pampulha e 7,95 no Buritis ($F = 0{,}07$ com 2 e 20 graus de liberdade; valor-p ≈ 0,94). **Não rejeitamos** $H_0$: os dados não mostram diferença de satisfação entre as unidades, e o $\eta^2$ de 0,007 indica que a unidade explica menos de 1% da variação das notas. Como não rejeitamos $H_0$, não há motivo para aplicar o teste de Tukey. Para a diretoria, o resultado sugere que o padrão de atendimento está homogêneo na rede; os esforços de melhoria podem ser definidos para a rede como um todo, e não para uma unidade específica. Lembre-se de que não rejeitar $H_0$ não prova que as médias sejam exatamente iguais.

---
## Exemplo 4. Perda de matéria-prima com quatro fornecedores

Uma indústria de embalagens compra bobinas de papelão de quatro fornecedores. O engenheiro de qualidade registrou o percentual de perda (material descartado no corte) em 5 lotes sorteados de cada fornecedor.

| Grupo | n | Valores |
|---|---|---|
| Fornecedor 1 | 5 | 2,1; 2,4; 1,9; 2,3; 2,0 |
| Fornecedor 2 | 5 | 2,8; 3,1; 2,6; 3,0; 2,9 |
| Fornecedor 3 | 5 | 2,2; 2,0; 2,5; 2,3; 2,1 |
| Fornecedor 4 | 5 | 3,4; 3,0; 3,3; 3,6; 3,2 |

**Pergunta gerencial.** A perda média de material é **diferente** entre os quatro fornecedores?

**Hipóteses** ($k = 4$ grupos):

* $H_0$: $\mu_{\text{1}} = \mu_{\text{2}} = \mu_{\text{3}} = \mu_{\text{4}}$ (os quatro fornecedores têm a mesma perda média)
* $H_1$: pelo menos uma média é diferente (pelo menos um fornecedor tem perda média diferente)

Teste F (unilateral à direita), com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em listas, uma para cada grupo, e mostrar a média de cada uma. Agora são quatro grupos; o raciocínio e o código são os mesmos, com uma lista a mais.

In [23]:
forn1 = [2.1, 2.4, 1.9, 2.3, 2.0]
forn2 = [2.8, 3.1, 2.6, 3.0, 2.9]
forn3 = [2.2, 2.0, 2.5, 2.3, 2.1]
forn4 = [3.4, 3.0, 3.3, 3.6, 3.2]

print("Média de forn1:", round(statistics.mean(forn1), 4))
print("Média de forn2:", round(statistics.mean(forn2), 4))
print("Média de forn3:", round(statistics.mean(forn3), 4))
print("Média de forn4:", round(statistics.mean(forn4), 4))

Média de forn1: 2.14
Média de forn2: 2.88
Média de forn3: 2.22
Média de forn4: 3.3


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo e a média geral. Para a média geral, juntamos todas as listas com o operador `+`.

In [24]:
n1 = len(forn1) # Tamanho do grupo 1.
n2 = len(forn2) # Tamanho do grupo 2.
n3 = len(forn3) # Tamanho do grupo 3.
n4 = len(forn4) # Tamanho do grupo 4.
media1 = statistics.mean(forn1) # Média do grupo 1.
media2 = statistics.mean(forn2) # Média do grupo 2.
media3 = statistics.mean(forn3) # Média do grupo 3.
media4 = statistics.mean(forn4) # Média do grupo 4.
dp1 = statistics.stdev(forn1) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(forn2) # Desvio-padrão amostral do grupo 2.
dp3 = statistics.stdev(forn3) # Desvio-padrão amostral do grupo 3.
dp4 = statistics.stdev(forn4) # Desvio-padrão amostral do grupo 4.

todos = forn1 + forn2 + forn3 + forn4 # Junta todas as observações.
N = len(todos) # Total de observações.
k = 4 # Número de grupos.
media_geral = statistics.mean(todos) # Média geral.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))
print("Grupo 3: n =", n3, "| média =", round(media3, 4), "| desvio-padrão =", round(dp3, 4))
print("Grupo 4: n =", n4, "| média =", round(media4, 4), "| desvio-padrão =", round(dp4, 4))
print("Média geral:", round(media_geral, 4), "| N =", N)

Grupo 1: n = 5 | média = 2.14 | desvio-padrão = 0.2074
Grupo 2: n = 5 | média = 2.88 | desvio-padrão = 0.1924
Grupo 3: n = 5 | média = 2.22 | desvio-padrão = 0.1924
Grupo 4: n = 5 | média = 3.3 | desvio-padrão = 0.2236
Média geral: 2.635 | N = 20


**Passo 3.** Calcular a estatística F à mão, usando as fórmulas da tabela da ANOVA.

In [25]:
sq_entre = n1 * (media1 - media_geral) ** 2 + n2 * (media2 - media_geral) ** 2 + n3 * (media3 - media_geral) ** 2 + n4 * (media4 - media_geral) ** 2 # Variação entre os grupos.
sq_dentro = (n1 - 1) * dp1 ** 2 + (n2 - 1) * dp2 ** 2 + (n3 - 1) * dp3 ** 2 + (n4 - 1) * dp4 ** 2 # Variação dentro dos grupos.
gl_entre = k - 1 # Graus de liberdade do numerador.
gl_dentro = N - k # Graus de liberdade do denominador.
qm_entre = sq_entre / gl_entre # Quadrado médio entre os grupos.
qm_dentro = sq_dentro / gl_dentro # Quadrado médio dentro dos grupos.
f_calculado = qm_entre / qm_dentro # Estatística F.
eta2 = sq_entre / (sq_entre + sq_dentro) # Proporção da variação explicada pelos grupos.

print("SQ entre:", round(sq_entre, 4), "| gl =", gl_entre, "| QM entre:", round(qm_entre, 4))
print("SQ dentro:", round(sq_dentro, 4), "| gl =", gl_dentro, "| QM dentro:", round(qm_dentro, 4))
print("F calculado:", round(f_calculado, 4))
print("Eta ao quadrado:", round(eta2, 4))

SQ entre: 4.5975 | gl = 3 | QM entre: 1.5325
SQ dentro: 0.668 | gl = 16 | QM dentro: 0.0417
F calculado: 36.7066
Eta ao quadrado: 0.8731


**Passo 4.** Conferir com a função `f_oneway`, que faz toda a conta de uma vez. Basta informar as listas, separadas por vírgulas. Valores-p muito pequenos aparecem em notação científica: `2.742e-05`, por exemplo, significa $2{,}742 \times 10^{-5}$, ou seja, 0,00002742.

In [26]:
resultado = stats.f_oneway(forn1, forn2, forn3, forn4)

print("F (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 8))

F (SciPy): 36.7066
valor-p: 2.1e-07


**Passo 5.** Verificar os pressupostos. No teste de Levene, $H_0$ é a igualdade das variâncias; no de Shapiro-Wilk, $H_0$ é a normalidade dos dados do grupo. Valores-p maiores que 0,05 indicam que não há evidência contra o pressuposto.

In [27]:
levene = stats.levene(forn1, forn2, forn3, forn4) # Homogeneidade das variâncias.
print("Levene: valor-p =", round(levene.pvalue, 4))

print("Shapiro-Wilk (forn1): valor-p =", round(stats.shapiro(forn1).pvalue, 4))
print("Shapiro-Wilk (forn2): valor-p =", round(stats.shapiro(forn2).pvalue, 4))
print("Shapiro-Wilk (forn3): valor-p =", round(stats.shapiro(forn3).pvalue, 4))
print("Shapiro-Wilk (forn4): valor-p =", round(stats.shapiro(forn4).pvalue, 4))

Levene: valor-p = 0.9861
Shapiro-Wilk (forn1): valor-p = 0.754
Shapiro-Wilk (forn2): valor-p = 0.9276
Shapiro-Wilk (forn3): valor-p = 0.9276
Shapiro-Wilk (forn4): valor-p = 0.9998


**Passo 6.** Comparar com o valor crítico e tomar a decisão.

In [28]:
alfa = 0.05
f_critico = stats.f.ppf(1 - alfa, gl_entre, gl_dentro) # O teste F é sempre unilateral à direita.
print("F crítico:", round(f_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0. Pelo menos uma média é diferente.")
else:
    print("Decisão: não rejeitamos H0.")

F crítico: 3.2389
Decisão: rejeitamos H0. Pelo menos uma média é diferente.


**Passo 7.** Como rejeitamos $H_0$, aplicamos o teste de Tukey para descobrir quais pares de grupos diferem. Na saída, os grupos são numerados a partir de zero, na ordem em que as listas foram informadas: 0 = `forn1`, 1 = `forn2`, 2 = `forn3`, 3 = `forn4`. Cada par aparece duas vezes, com sinais trocados; basta ler uma das linhas. A coluna `p-value` mostra três casas decimais, e `0.000` indica valor-p menor que 0,0005.

In [29]:
tukey = stats.tukey_hsd(forn1, forn2, forn3, forn4)
print(tukey)

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.740     0.000    -1.110    -0.370
 (0 - 2)     -0.080     0.924    -0.450     0.290
 (0 - 3)     -1.160     0.000    -1.530    -0.790
 (1 - 0)      0.740     0.000     0.370     1.110
 (1 - 2)      0.660     0.001     0.290     1.030
 (1 - 3)     -0.420     0.023    -0.790    -0.050
 (2 - 0)      0.080     0.924    -0.290     0.450
 (2 - 1)     -0.660     0.001    -1.030    -0.290
 (2 - 3)     -1.080     0.000    -1.450    -0.710
 (3 - 0)      1.160     0.000     0.790     1.530
 (3 - 1)      0.420     0.023     0.050     0.790
 (3 - 2)      1.080     0.000     0.710     1.450



**Leitura do Tukey.** Com quatro grupos, há seis pares. Só o par fornecedor 1 e fornecedor 3 (grupos 0 e 2) não difere (diferença de 0,08 ponto percentual; valor-p ≈ 0,92). Os demais pares têm valores-p abaixo de 0,05; o menos nítido é o par 2 e 4 (diferença de 0,42 ponto; valor-p ≈ 0,023).

**Interpretação gerencial.** As perdas médias foram de 2,14% (fornecedor 1), 2,88% (fornecedor 2), 2,22% (fornecedor 3) e 3,30% (fornecedor 4), com $F = 36{,}71$ (3 e 16 graus de liberdade) e valor-p < 0,000001. Rejeitamos $H_0$. O teste de Tukey separa os fornecedores em três patamares: os fornecedores 1 e 3, com as menores perdas e sem diferença entre si; o fornecedor 2, intermediário; e o fornecedor 4, com a maior perda. Para o setor de compras, os fornecedores 1 e 3 são equivalentes em qualidade, e a escolha entre eles pode ser feita com base em preço e prazo. A diferença de cerca de 1,1 ponto percentual entre os fornecedores 1 e 4 deve ser convertida em reais, considerando o volume anual comprado, para avaliar o ganho de trocar de fornecedor.

---
## Exemplo 5. Quatro métodos de treinamento de vendedores

Uma corretora de seguros testou quatro métodos de treinamento para novos vendedores: presencial, a distância, mentoria individual e um formato misto. Seis vendedores foram sorteados para cada método, e foi registrado o número de contratos fechados por cada um no primeiro mês após o treinamento.

| Grupo | n | Valores |
|---|---|---|
| Presencial | 6 | 18; 22; 20; 17; 23; 19 |
| A distância | 6 | 21; 19; 24; 20; 22; 18 |
| Mentoria | 6 | 20; 23; 19; 25; 21; 22 |
| Misto | 6 | 17; 21; 19; 22; 20; 18 |

**Pergunta gerencial.** O número médio de contratos fechados é **diferente** entre os quatro métodos?

**Hipóteses** ($k = 4$ grupos):

* $H_0$: $\mu_{\text{presencial}} = \mu_{\text{distância}} = \mu_{\text{mentoria}} = \mu_{\text{misto}}$ (os quatro métodos produzem o mesmo resultado médio)
* $H_1$: pelo menos uma média é diferente (pelo menos um método produz resultado médio diferente)

Teste F (unilateral à direita), com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em listas, uma para cada grupo, e mostrar a média de cada uma.

In [30]:
presencial = [18, 22, 20, 17, 23, 19]
distancia = [21, 19, 24, 20, 22, 18]
mentoria = [20, 23, 19, 25, 21, 22]
misto = [17, 21, 19, 22, 20, 18]

print("Média de presencial:", round(statistics.mean(presencial), 4))
print("Média de distancia:", round(statistics.mean(distancia), 4))
print("Média de mentoria:", round(statistics.mean(mentoria), 4))
print("Média de misto:", round(statistics.mean(misto), 4))

Média de presencial: 19.8333
Média de distancia: 20.6667
Média de mentoria: 21.6667
Média de misto: 19.5


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo e a média geral. Para a média geral, juntamos todas as listas com o operador `+`.

In [31]:
n1 = len(presencial) # Tamanho do grupo 1.
n2 = len(distancia) # Tamanho do grupo 2.
n3 = len(mentoria) # Tamanho do grupo 3.
n4 = len(misto) # Tamanho do grupo 4.
media1 = statistics.mean(presencial) # Média do grupo 1.
media2 = statistics.mean(distancia) # Média do grupo 2.
media3 = statistics.mean(mentoria) # Média do grupo 3.
media4 = statistics.mean(misto) # Média do grupo 4.
dp1 = statistics.stdev(presencial) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(distancia) # Desvio-padrão amostral do grupo 2.
dp3 = statistics.stdev(mentoria) # Desvio-padrão amostral do grupo 3.
dp4 = statistics.stdev(misto) # Desvio-padrão amostral do grupo 4.

todos = presencial + distancia + mentoria + misto # Junta todas as observações.
N = len(todos) # Total de observações.
k = 4 # Número de grupos.
media_geral = statistics.mean(todos) # Média geral.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))
print("Grupo 3: n =", n3, "| média =", round(media3, 4), "| desvio-padrão =", round(dp3, 4))
print("Grupo 4: n =", n4, "| média =", round(media4, 4), "| desvio-padrão =", round(dp4, 4))
print("Média geral:", round(media_geral, 4), "| N =", N)

Grupo 1: n = 6 | média = 19.8333 | desvio-padrão = 2.3166
Grupo 2: n = 6 | média = 20.6667 | desvio-padrão = 2.1602
Grupo 3: n = 6 | média = 21.6667 | desvio-padrão = 2.1602
Grupo 4: n = 6 | média = 19.5 | desvio-padrão = 1.8708
Média geral: 20.4167 | N = 24


**Passo 3.** Calcular a estatística F à mão, usando as fórmulas da tabela da ANOVA.

In [32]:
sq_entre = n1 * (media1 - media_geral) ** 2 + n2 * (media2 - media_geral) ** 2 + n3 * (media3 - media_geral) ** 2 + n4 * (media4 - media_geral) ** 2 # Variação entre os grupos.
sq_dentro = (n1 - 1) * dp1 ** 2 + (n2 - 1) * dp2 ** 2 + (n3 - 1) * dp3 ** 2 + (n4 - 1) * dp4 ** 2 # Variação dentro dos grupos.
gl_entre = k - 1 # Graus de liberdade do numerador.
gl_dentro = N - k # Graus de liberdade do denominador.
qm_entre = sq_entre / gl_entre # Quadrado médio entre os grupos.
qm_dentro = sq_dentro / gl_dentro # Quadrado médio dentro dos grupos.
f_calculado = qm_entre / qm_dentro # Estatística F.
eta2 = sq_entre / (sq_entre + sq_dentro) # Proporção da variação explicada pelos grupos.

print("SQ entre:", round(sq_entre, 4), "| gl =", gl_entre, "| QM entre:", round(qm_entre, 4))
print("SQ dentro:", round(sq_dentro, 4), "| gl =", gl_dentro, "| QM dentro:", round(qm_dentro, 4))
print("F calculado:", round(f_calculado, 4))
print("Eta ao quadrado:", round(eta2, 4))

SQ entre: 16.8333 | gl = 3 | QM entre: 5.6111
SQ dentro: 91.0 | gl = 20 | QM dentro: 4.55
F calculado: 1.2332
Eta ao quadrado: 0.1561


**Passo 4.** Conferir com a função `f_oneway`, que faz toda a conta de uma vez. Basta informar as listas, separadas por vírgulas. Valores-p muito pequenos aparecem em notação científica: `2.742e-05`, por exemplo, significa $2{,}742 \times 10^{-5}$, ou seja, 0,00002742.

In [33]:
resultado = stats.f_oneway(presencial, distancia, mentoria, misto)

print("F (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 8))

F (SciPy): 1.2332
valor-p: 0.32382446


**Passo 5.** Verificar os pressupostos. No teste de Levene, $H_0$ é a igualdade das variâncias; no de Shapiro-Wilk, $H_0$ é a normalidade dos dados do grupo. Valores-p maiores que 0,05 indicam que não há evidência contra o pressuposto.

In [34]:
levene = stats.levene(presencial, distancia, mentoria, misto) # Homogeneidade das variâncias.
print("Levene: valor-p =", round(levene.pvalue, 4))

print("Shapiro-Wilk (presencial): valor-p =", round(stats.shapiro(presencial).pvalue, 4))
print("Shapiro-Wilk (distancia): valor-p =", round(stats.shapiro(distancia).pvalue, 4))
print("Shapiro-Wilk (mentoria): valor-p =", round(stats.shapiro(mentoria).pvalue, 4))
print("Shapiro-Wilk (misto): valor-p =", round(stats.shapiro(misto).pvalue, 4))

Levene: valor-p = 0.9653
Shapiro-Wilk (presencial): valor-p = 0.8006
Shapiro-Wilk (distancia): valor-p = 0.9637
Shapiro-Wilk (mentoria): valor-p = 0.9637
Shapiro-Wilk (misto): valor-p = 0.9606


**Passo 6.** Comparar com o valor crítico e tomar a decisão.

In [35]:
alfa = 0.05
f_critico = stats.f.ppf(1 - alfa, gl_entre, gl_dentro) # O teste F é sempre unilateral à direita.
print("F crítico:", round(f_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0. Pelo menos uma média é diferente.")
else:
    print("Decisão: não rejeitamos H0.")

F crítico: 3.0984
Decisão: não rejeitamos H0.


**Interpretação gerencial.** As médias foram de 19,83 contratos (presencial), 20,67 (a distância), 21,67 (mentoria) e 19,50 (misto), com $F = 1{,}23$ (3 e 20 graus de liberdade) e valor-p ≈ 0,32. **Não rejeitamos** $H_0$: as diferenças entre os métodos são pequenas diante da variação entre os vendedores de um mesmo método. A mentoria teve a maior média, mas isso não basta para afirmar que ela é superior. O $\eta^2$ de 0,16 mostra que os métodos explicam parte da variação, mas, com apenas seis vendedores por grupo, a amostra é pequena para separar um efeito real do acaso. Como a mentoria individual costuma ser o método mais caro, a corretora não tem, por enquanto, justificativa estatística para adotá-la; pode manter o método mais barato e repetir a avaliação com mais vendedores.

---
## Exemplo 6. Produtividade em três turnos

Uma fábrica de autopeças funciona em três turnos e quer saber se a produtividade da linha de montagem varia ao longo do dia. Em 6 dias sorteados, foi registrado o número de peças montadas por hora em cada turno. Como os operadores e as condições de cada turno são diferentes, tratamos os turnos como grupos independentes.

| Grupo | n | Valores |
|---|---|---|
| Manhã | 6 | 120; 126; 119; 117; 125; 122 |
| Tarde | 6 | 114; 121; 117; 116; 115; 122 |
| Noite | 6 | 116; 121; 114; 121; 118; 115 |

**Pergunta gerencial.** A produtividade média é **diferente** entre os três turnos?

**Hipóteses** ($k = 3$ grupos):

* $H_0$: $\mu_{\text{manhã}} = \mu_{\text{tarde}} = \mu_{\text{noite}}$ (os três turnos têm a mesma produtividade média)
* $H_1$: pelo menos uma média é diferente (pelo menos um turno tem produtividade média diferente)

Teste F (unilateral à direita), com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em listas, uma para cada grupo, e mostrar a média de cada uma.

In [36]:
manha = [120, 126, 119, 117, 125, 122]
tarde = [114, 121, 117, 116, 115, 122]
noite = [116, 121, 114, 121, 118, 115]

print("Média de manha:", round(statistics.mean(manha), 4))
print("Média de tarde:", round(statistics.mean(tarde), 4))
print("Média de noite:", round(statistics.mean(noite), 4))

Média de manha: 121.5
Média de tarde: 117.5
Média de noite: 117.5


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo e a média geral. Para a média geral, juntamos todas as listas com o operador `+`.

In [37]:
n1 = len(manha) # Tamanho do grupo 1.
n2 = len(tarde) # Tamanho do grupo 2.
n3 = len(noite) # Tamanho do grupo 3.
media1 = statistics.mean(manha) # Média do grupo 1.
media2 = statistics.mean(tarde) # Média do grupo 2.
media3 = statistics.mean(noite) # Média do grupo 3.
dp1 = statistics.stdev(manha) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(tarde) # Desvio-padrão amostral do grupo 2.
dp3 = statistics.stdev(noite) # Desvio-padrão amostral do grupo 3.

todos = manha + tarde + noite # Junta todas as observações.
N = len(todos) # Total de observações.
k = 3 # Número de grupos.
media_geral = statistics.mean(todos) # Média geral.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))
print("Grupo 3: n =", n3, "| média =", round(media3, 4), "| desvio-padrão =", round(dp3, 4))
print("Média geral:", round(media_geral, 4), "| N =", N)

Grupo 1: n = 6 | média = 121.5 | desvio-padrão = 3.5071
Grupo 2: n = 6 | média = 117.5 | desvio-padrão = 3.2711
Grupo 3: n = 6 | média = 117.5 | desvio-padrão = 3.0166
Média geral: 118.8333 | N = 18


**Passo 3.** Calcular a estatística F à mão, usando as fórmulas da tabela da ANOVA.

In [38]:
sq_entre = n1 * (media1 - media_geral) ** 2 + n2 * (media2 - media_geral) ** 2 + n3 * (media3 - media_geral) ** 2 # Variação entre os grupos.
sq_dentro = (n1 - 1) * dp1 ** 2 + (n2 - 1) * dp2 ** 2 + (n3 - 1) * dp3 ** 2 # Variação dentro dos grupos.
gl_entre = k - 1 # Graus de liberdade do numerador.
gl_dentro = N - k # Graus de liberdade do denominador.
qm_entre = sq_entre / gl_entre # Quadrado médio entre os grupos.
qm_dentro = sq_dentro / gl_dentro # Quadrado médio dentro dos grupos.
f_calculado = qm_entre / qm_dentro # Estatística F.
eta2 = sq_entre / (sq_entre + sq_dentro) # Proporção da variação explicada pelos grupos.

print("SQ entre:", round(sq_entre, 4), "| gl =", gl_entre, "| QM entre:", round(qm_entre, 4))
print("SQ dentro:", round(sq_dentro, 4), "| gl =", gl_dentro, "| QM dentro:", round(qm_dentro, 4))
print("F calculado:", round(f_calculado, 4))
print("Eta ao quadrado:", round(eta2, 4))

SQ entre: 64.0 | gl = 2 | QM entre: 32.0
SQ dentro: 160.5 | gl = 15 | QM dentro: 10.7
F calculado: 2.9907
Eta ao quadrado: 0.2851


**Passo 4.** Conferir com a função `f_oneway`, que faz toda a conta de uma vez. Basta informar as listas, separadas por vírgulas. Valores-p muito pequenos aparecem em notação científica: `2.742e-05`, por exemplo, significa $2{,}742 \times 10^{-5}$, ou seja, 0,00002742.

In [39]:
resultado = stats.f_oneway(manha, tarde, noite)

print("F (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 8))

F (SciPy): 2.9907
valor-p: 0.08071239


**Passo 5.** Verificar os pressupostos. No teste de Levene, $H_0$ é a igualdade das variâncias; no de Shapiro-Wilk, $H_0$ é a normalidade dos dados do grupo. Valores-p maiores que 0,05 indicam que não há evidência contra o pressuposto.

In [40]:
levene = stats.levene(manha, tarde, noite) # Homogeneidade das variâncias.
print("Levene: valor-p =", round(levene.pvalue, 4))

print("Shapiro-Wilk (manha): valor-p =", round(stats.shapiro(manha).pvalue, 4))
print("Shapiro-Wilk (tarde): valor-p =", round(stats.shapiro(tarde).pvalue, 4))
print("Shapiro-Wilk (noite): valor-p =", round(stats.shapiro(noite).pvalue, 4))

Levene: valor-p = 0.9309
Shapiro-Wilk (manha): valor-p = 0.7481
Shapiro-Wilk (tarde): valor-p = 0.3403
Shapiro-Wilk (noite): valor-p = 0.3153


**Passo 6.** Comparar com o valor crítico e tomar a decisão.

In [41]:
alfa = 0.05
f_critico = stats.f.ppf(1 - alfa, gl_entre, gl_dentro) # O teste F é sempre unilateral à direita.
print("F crítico:", round(f_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0. Pelo menos uma média é diferente.")
else:
    print("Decisão: não rejeitamos H0.")

F crítico: 3.6823
Decisão: não rejeitamos H0.


**Interpretação gerencial.** O turno da manhã montou, em média, 121,5 peças por hora, e os turnos da tarde e da noite, 117,5 cada ($F = 2{,}99$ com 2 e 15 graus de liberdade; valor-p ≈ 0,081). Ao nível de 5%, **não rejeitamos** $H_0$, embora o resultado seja limítrofe. Se a fábrica tivesse adotado $\alpha = 10\%$, a conclusão seria outra, o que reforça que o nível de significância deve ser definido **antes** da análise. Como não rejeitamos $H_0$, não aplicamos o teste de Tukey. A recomendação é ampliar a coleta para mais dias antes de investigar as causas de uma possível queda de produtividade à tarde e à noite.

---
## Para ir além: a mesma análise com a biblioteca statsmodels

Em análises com mais dados, é comum organizar as observações em uma tabela com uma coluna para a variável medida e outra para o grupo. A biblioteca **pandas** cria essa tabela (`DataFrame`), e a biblioteca **statsmodels** produz a tabela da ANOVA completa. Vamos refazer o Exemplo 1 dessa forma. Esta seção é opcional e serve para mostrar como a ANOVA aparece em relatórios profissionais.

A fórmula `'vendas ~ C(promocao)'` significa "explicar as vendas pela promoção", e `C()` indica que `promocao` é uma variável categórica (de grupos).

In [42]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

dados = pd.DataFrame({
    'vendas': desconto + leve3 + brinde,
    'promocao': ['desconto'] * 6 + ['leve3'] * 7 + ['brinde'] * 6
})

modelo = ols('vendas ~ C(promocao)', data=dados).fit() # Ajusta o modelo da ANOVA.
tabela_anova = sm.stats.anova_lm(modelo, typ=2) # Monta a tabela da ANOVA.
print(tabela_anova)

                 sum_sq    df          F    PR(>F)
C(promocao)  284.932331   2.0  21.738706  0.000027
Residual     104.857143  16.0        NaN       NaN


A linha `C(promocao)` traz a soma de quadrados entre os grupos (`sum_sq`), os graus de liberdade (`df`), o F e o valor-p (`PR(>F)`); a linha `Residual` traz a soma de quadrados dentro dos grupos. Os números são os mesmos que calculamos à mão no Exemplo 1.

Com o modelo ajustado, também podemos testar a normalidade dos **resíduos** (a diferença entre cada valor e a média do seu grupo), que é a forma recomendada nos livros de planejamento de experimentos, e aplicar o teste de Tukey com nomes em vez de números.

In [43]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print("Shapiro-Wilk dos resíduos: valor-p =", round(stats.shapiro(modelo.resid).pvalue, 4))
print(pairwise_tukeyhsd(dados['vendas'], dados['promocao'], alpha=0.05))

Shapiro-Wilk dos resíduos: valor-p = 0.5358
  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1   group2  meandiff p-adj   lower   upper  reject
--------------------------------------------------------
  brinde desconto      2.0 0.3878 -1.8138  5.8138  False
  brinde    leve3   8.8571    0.0  5.1821 12.5322   True
desconto    leve3   6.8571 0.0005  3.1821 10.5322   True
--------------------------------------------------------


**E se as variâncias forem diferentes?** Quando o teste de Levene rejeita a igualdade das variâncias, sobretudo com grupos de tamanhos diferentes, a ANOVA tradicional pode errar a decisão. Nesse caso, uma alternativa é a **ANOVA de Welch**, que, assim como o teste t de Welch da aula passada, não supõe variâncias iguais. Na statsmodels, ela é obtida com `anova_oneway(..., use_var='unequal')`. Quando a normalidade é claramente violada, a alternativa não paramétrica é o teste de Kruskal-Wallis (`stats.kruskal`).

In [44]:
from statsmodels.stats.oneway import anova_oneway

welch = anova_oneway(dados['vendas'], dados['promocao'], use_var='unequal')
print("ANOVA de Welch: F =", round(welch.statistic, 4), "| valor-p =", round(welch.pvalue, 6))

ANOVA de Welch: F = 22.7448 | valor-p = 0.000163


---
## Síntese da aula

| Exemplo | k | F | gl | valor-p | $\eta^2$ | Decisão ($\alpha$ = 5%) | Tukey |
|---|---|---|---|---|---|---|---|
| 1. Formatos de promoção | 3 | 21,74 | 2 e 16 | 0,00003 | 0,73 | rejeita $H_0$ | leve 3, pague 2 supera os demais |
| 2. Transportadoras | 3 | 26,69 | 2 e 17 | 0,000006 | 0,76 | rejeita $H_0$ | os três pares diferem |
| 3. Unidades do restaurante | 3 | 0,07 | 2 e 20 | 0,94 | 0,01 | não rejeita $H_0$ | não se aplica |
| 4. Fornecedores de papelão | 4 | 36,71 | 3 e 16 | < 0,000001 | 0,87 | rejeita $H_0$ | só 1 e 3 não diferem |
| 5. Métodos de treinamento | 4 | 1,23 | 3 e 20 | 0,32 | 0,16 | não rejeita $H_0$ | não se aplica |
| 6. Turnos de produção | 3 | 2,99 | 2 e 15 | 0,081 | 0,29 | não rejeita $H_0$ | não se aplica |

Em todos os exemplos, os testes de Levene e de Shapiro-Wilk tiveram valores-p acima de 0,05.

**Qual teste usar?**

* dois grupos independentes: teste t para amostras independentes (`ttest_ind`);
* dois grupos pareados: teste t pareado (`ttest_rel`);
* três ou mais grupos independentes: ANOVA com um fator (`f_oneway`), seguida do teste de Tukey quando $H_0$ for rejeitada.

**Roteiro para qualquer ANOVA com um fator em Python:**

1. digite os dados em listas, uma para cada grupo;
2. escreva as hipóteses ($H_0$: todas as médias são iguais; $H_1$: pelo menos uma é diferente) e fixe $\alpha$ antes de olhar o resultado;
3. use `stats.f_oneway(grupo1, grupo2, grupo3, ...)`;
4. verifique os pressupostos com `stats.levene(...)` e `stats.shapiro(...)`;
5. compare o valor-p com $\alpha$ e, se rejeitar $H_0$, aplique `stats.tukey_hsd(...)`;
6. traduza a decisão estatística em uma recomendação de negócios, considerando custos e tamanho do efeito.

## Exercícios propostos

**Exercício 1.** Uma operadora de planos de saúde avalia três equipes de atendimento telefônico. A supervisora registrou o tempo de atendimento, em minutos, de 6 ligações sorteadas de cada equipe.

* Equipe Alfa: 12,5; 14,0; 13,2; 15,1; 12,8; 13,9
* Equipe Beta: 16,2; 15,4; 17,8; 16,9; 15,1; 17,2
* Equipe Gama: 13,6; 14,8; 12,9; 15,5; 14,1; 13,3

Ao nível de 5%, o tempo médio de atendimento é diferente entre as equipes? Verifique os pressupostos e, se for o caso, identifique com o teste de Tukey quais equipes diferem. Redija uma recomendação para a supervisora.

**Exercício 2.** Uma loja virtual testou três versões da página inicial e pediu a visitantes sorteados que dessem uma nota de 0 a 10 para a facilidade de navegação.

* Versão A: 7,8; 8,2; 7,5; 8,0; 7,9; 8,4; 7,6
* Versão B: 8,1; 7,7; 8,3; 7,9; 8,0; 7,6; 8,2
* Versão C: 7,9; 8,0; 7,6; 8,3; 7,8; 8,1; 7,7

Ao nível de 5%, há diferença na nota média das três versões?

Use as células abaixo para resolver.

In [45]:
# Exercício 1

In [46]:
# Exercício 2

## Referências

ANDERSON, D. R.; SWEENEY, D. J.; WILLIAMS, T. A.; CAMM, J. D.; COCHRAN, J. J. *Estatística aplicada à administração e economia*. São Paulo: Cengage Learning. Capítulo sobre experimentos e análise de variância.

MONTGOMERY, D. C. *Design and analysis of experiments*. 9. ed. Hoboken: John Wiley & Sons, 2017.

TUKEY, J. W. Comparing individual means in the analysis of variance. *Biometrics*, v. 5, n. 2, p. 99–114, 1949.

SHAPIRO, S. S.; WILK, M. B. An analysis of variance test for normality (complete samples). *Biometrika*, v. 52, n. 3/4, p. 591–611, 1965.

BROWN, M. B.; FORSYTHE, A. B. Robust tests for the equality of variances. *Journal of the American Statistical Association*, v. 69, n. 346, p. 364–367, 1974.

WELCH, B. L. On the comparison of several mean values: an alternative approach. *Biometrika*, v. 38, n. 3/4, p. 330–336, 1951.

VIRTANEN, P. et al. SciPy 1.0: fundamental algorithms for scientific computing in Python. *Nature Methods*, v. 17, p. 261–272, 2020.

SEABOLD, S.; PERKTOLD, J. Statsmodels: econometric and statistical modeling with Python. In: PYTHON IN SCIENCE CONFERENCE, 9., 2010, Austin. *Proceedings* [...]. Austin: SciPy, 2010. p. 92–96.

SCIPY. *scipy.stats.f_oneway*; *scipy.stats.levene*; *scipy.stats.shapiro*; *scipy.stats.tukey_hsd*. Documentação oficial. Disponível em: https://docs.scipy.org/doc/scipy/reference/stats.html.

STATSMODELS. *ANOVA*; *statsmodels.stats.oneway.anova_oneway*; *statsmodels.stats.multicomp.pairwise_tukeyhsd*. Documentação oficial. Disponível em: https://www.statsmodels.org/stable/anova.html.

**Observação sobre o teste de Levene.** Na SciPy, `stats.levene` usa por padrão a mediana de cada grupo (`center='median'`), variante proposta por Brown e Forsythe (1974), mais robusta quando os dados não são normais.